# Mini-TP 6 — Tu modelo en el Data Lake

Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual

El mismo modelo de arrestos de Chicago que sirven las APIs del repo, ahora guardado en un Data Lake y servido desde ahí en lugar de leerlo del disco.

## Qué se entrega

1. Un bucket con las zonas `raw`, `curated` y `models`.
2. El artefacto del modelo versionado en `models/v1/`.
3. Una función que carga el modelo desde el lake y predice.
4. El experimento registrado en MLflow, con los artefactos en el lake.
5. La reflexión.

Como en los TPs anteriores, los `.py` de `tp6_datalake/` son la fuente y están cubiertos por tests; el notebook los importa y los muestra.

Necesita MinIO levantado (`make install` del compose del repo, o el contenedor suelto que indica la consigna) y, para el paso opcional, MLflow.

In [1]:
import os
from pathlib import Path

import arrest_model

# El notebook vive en tp6_datalake/, pero los imports esperan la raíz del repo.
REPO = Path(arrest_model.__file__).resolve().parent.parent
os.chdir(REPO)

from tp6_datalake.lake import ENDPOINT, lake_available
from tp6_datalake.tracking import TRACKING_URI, tracking_available

print(f"MinIO en {ENDPOINT}: {'disponible' if lake_available() else 'no responde'}")
print(f"MLflow en {TRACKING_URI}: {'disponible' if tracking_available() else 'no responde'}")

MinIO en http://localhost:9000: disponible
MLflow en http://localhost:5001: disponible


## 1. El bucket y sus zonas

El lake es MinIO, que habla la misma API que S3. Lo que vale de eso es que el código no cambia si mañana el bucket vive en AWS: cambian el endpoint y las credenciales.

Las zonas no son carpetas. En S3 las claves son planas y la barra es una convención, así que `raw/crimes/dia=2026-10-03/crimes.csv` es una clave entera y la zona es su prefijo. Separarlas es lo que evita el pantano: en `raw` está lo que llegó tal cual, en `curated` lo que ya está listo para entrenar, y en `models` los artefactos versionados.

In [2]:
from tp6_datalake.lake import BUCKET, ZONES, client, ensure_bucket, list_keys

s3 = client()
ensure_bucket(s3, BUCKET)
print(f"bucket {BUCKET} con zonas: {', '.join(ZONES)}")
print(list_keys(s3, "", BUCKET))

bucket arrest-lake con zonas: raw, curated, models
['curated/.keep', 'curated/arrests/arrests.parquet', 'models/.keep', 'models/v1/model.pkl', 'models/v2/model.pkl', 'raw/.keep', 'raw/crimes/dia=2026-10-03/crimes.csv', 'raw/police_stations/police_stations.csv']


## 2. La zona raw: lo que llega de Socrata

Dos fuentes con cadencias muy distintas. Los reportes de crímenes se actualizan a diario, así que se particionan por día. Las comisarías son 23 filas que no cambiaron desde 2016: se reemplazan enteras y sin partición.

Si el portal no responde —sin red, sin token o con el límite de uso agotado— se usa una copia versionada en el repo. Es a propósito: un pipeline que solo funciona con internet no se puede corregir en la máquina de otro.

In [3]:
from datetime import UTC, datetime

from tp6_datalake.ingest import land_raw

dia = datetime.now(UTC).date().isoformat()
aterrizado = land_raw(s3, bucket=BUCKET, day=dia)
print("origen de los datos:", aterrizado["origen"])
for clave in ("crimes", "stations"):
    print(" ", aterrizado[clave])

origen de los datos: socrata
  s3://arrest-lake/raw/crimes/dia=2026-10-03/crimes.csv
  s3://arrest-lake/raw/police_stations/police_stations.csv


## 3. La zona curated: el dataset listo para entrenar

En `raw` están las 22 columnas crudas con sus huecos. En `curated` van las 7 features que el modelo consume y la etiqueta, en Parquet: guarda los tipos, se comprime por columna y permite leer una columna sin recorrer el resto.

In [4]:
import io

import pandas as pd

from tp6_datalake.curated import land_curated
from tp6_datalake.lake import get_bytes

print(land_curated(s3, bucket=BUCKET))
curado = pd.read_parquet(io.BytesIO(get_bytes(s3, "curated/arrests/arrests.parquet", BUCKET)))
print(f"{len(curado)} filas · {len(curado.columns)} columnas")
curado.head(3)

s3://arrest-lake/curated/arrests/arrests.parquet
20000 filas · 8 columnas


,IUCR_freq,Primary_Type_freq,Location_Description_freq,Day_sin,X Coordinate_standardized,Y Coordinate_standardized,Distance Crime To Police Station_standardized,Arrest_tag
0,0.018222,0.091165,0.273356,-0.433884,0.022256,0.510297,0.549988,1
1,0.001112,0.179677,0.273356,0.974928,1.094675,-1.207130,0.780269,0
2,0.062346,0.179677,0.054023,0.781831,1.094983,-0.830385,-1.081973,1


## 4. La zona models, y el corazón del TP: servir desde el lake

El modelo se publica bajo su versión y se carga desde el lake para predecir, sin pasar por disco.

La versión va en el prefijo, así conviven varias y volver atrás es apuntar a la anterior en lugar de reconstruir nada.

In [5]:
from tp6_datalake.models import load_model, publish_model

print(publish_model(s3, bucket=BUCKET, version="v1"))
print(publish_model(s3, bucket=BUCKET, version="v2"))
print("modelos en el lake:", list_keys(s3, "models/", BUCKET))

s3://arrest-lake/models/v1/model.pkl
s3://arrest-lake/models/v2/model.pkl
modelos en el lake: ['models/.keep', 'models/v1/model.pkl', 'models/v2/model.pkl']


In [6]:
from IPython.display import Code, display

display(Code(filename="tp6_datalake/models.py", language="python"))

"""Mini-TP 6: el modelo publicado en el lake y servido desde ahí.

Hasta acá, los tres servicios del repo copian `model/model.pkl` adentro de su imagen. Eso ata
el modelo al contenedor: publicar uno nuevo obliga a reconstruir y redesplegar las tres
imágenes, y nada garantiza que las tres estén sirviendo la misma versión.

Con el modelo en el lake, la imagen deja de contenerlo y pasa a saber dónde buscarlo. Publicar
una versión nueva es subir un objeto, y el servicio la toma al arrancar.

La versión va en el prefijo, `models/v1/`, así conviven varias y volver atrás es apuntar a la
anterior en vez de reconstruir nada.
"""

import io
from pathlib import Path
from typing import Any

import joblib
from botocore.exceptions import ClientError

from arrest_model.model import MODEL_PATH
from tp6_datalake.lake import BUCKET, get_bytes, put_bytes

VERSION = "v1"


def model_key(version: str = VERSION) -> str:
    """Dónde vive esa versión del modelo dentro del lake."""
    return f"models/{version}/model.pkl"


def publish_model(
    s3: Any, bucket: str = BUCKET, version: str = VERSION, path: Path = MODEL_PATH
) -> str:
    """Sube el bundle entrenado a la zona de modelos, bajo su versión."""
    return put_bytes(s3, model_key(version), path.read_bytes(), bucket)


def load_model(s3: Any, bucket: str = BUCKET, version: str = VERSION) -> dict[str, Any]:
    """Trae el bundle desde el lake y lo reconstruye en memoria, sin tocar el disco."""
    try:
        crudo = get_bytes(s3, model_key(version), bucket)
    except ClientError as error:
        raise FileNotFoundError(
            f"No hay modelo {version} en s3://{bucket}/{model_key(version)}"
        ) from error
    return joblib.load(io.BytesIO(crudo))

In [7]:
from arrest_model.model import load_bundle, predict
from arrest_model.schemas import EXAMPLE_REPORT, CrimeReport
from tp4_streaming.events import crime_stream

reporte = CrimeReport.model_validate(EXAMPLE_REPORT)
desde_el_lake = load_model(s3, bucket=BUCKET, version="v1")
print("predicción con el modelo del lake:", predict(desde_el_lake, [reporte])[0])

# El invariante: lo que viajó al lake y volvió predice igual que el .pkl del disco.
reportes = [CrimeReport.model_validate(e) for e in crime_stream(200, seed=4)]
iguales = predict(desde_el_lake, reportes) == predict(load_bundle(), reportes)
print(f"las {len(reportes)} predicciones coinciden con las del modelo local:", iguales)

predicción con el modelo del lake: arrest=0 probability=0.06843266636133194 model_name='chicago-arrest-xgboost' model_version=1
las 200 predicciones coinciden con las del modelo local: True


## 5. El experimento en MLflow, con los artefactos en el lake

MLflow parte el problema en dos: la metadata de la corrida —parámetros, métricas, etiquetas— va a PostgreSQL, y los artefactos pesados van al lake. Por eso el servidor se levanta apuntando a `s3://mlflow/`.

In [8]:
from tp6_datalake.tracking import artifacts_of, log_run, metrics_of

if tracking_available():
    corrida = log_run()
    print("corrida:", corrida)
    print("métricas registradas:", {k: round(v, 4) for k, v in metrics_of(corrida).items()})
    print("artefactos:", artifacts_of(corrida))
    print("en el lake:", [k for k in list_keys(s3, "", "mlflow") if corrida in k])
else:
    print("No hay servidor de MLflow en 5001; este paso es el opcional de la consigna.")

🏃 View run dashing-midge-292 at: http://localhost:5001/#/experiments/2/runs/beb0c9bd9e1541ad918760f191b34e10
🧪 View experiment at: http://localhost:5001/#/experiments/2
corrida: beb0c9bd9e1541ad918760f191b34e10
métricas registradas: {'accuracy': 0.9119, 'precision': 0.9047, 'recall': 0.9119, 'f1': 0.9039, 'auc': 0.8861, 'mcc': 0.5811}
artefactos: ['model.pkl']
en el lake: ['2/beb0c9bd9e1541ad918760f191b34e10/artifacts/model.pkl']


### Reflexión

> Pendiente de completar: por qué servir desde el lake, cómo versionar sin romper producción, y en qué zona guardar las predicciones.